Here is a comprehensive Python cheat sheet tailored for data analytics and data engineering, covering your requested topics along with essential foundational concepts.

---

## 1. Core Foundations: Loops, Lists vs. Arrays

### Loops

Data engineers use loops for automation, file orchestration, and data transformation pipelines.

In [ ]:
# For Loop: Iterating over a sequence (e.g., a list of file paths)
files = ['sales_01.csv', 'sales_02.csv', 'sales_03.csv']
for file in files:
    print(f"Processing database ingestion for: {file}")

# While Loop: Running based on a condition (e.g., API polling with backoff)
attempts = 0
while attempts < 3:
    print("Polling API for new data...")
    attempts += 1

# List Comprehension: Concise way to create/transform lists (Highly Optimized)
uppercase_files = [file.upper() for file in files]
# Result: ['SALES_01.CSV', 'SALES_02.CSV', 'SALES_03.CSV']

### Lists vs. Arrays

* **Python Lists:** Built-in, can hold mixed data types, dynamically sized, but slow for mathematical operations.
* **NumPy Arrays:** Require an import, homogenous data types (all integers, floats, etc.), fixed size on creation, and highly optimized for vectorization and mathematical operations.

In [ ]:
import numpy as np

# --- 1. Python Lists ---
# Instantiation
my_list = [1, "data", 3.14, True]
# Appending
my_list.append("new_element")

# --- 2. NumPy Arrays ---
# Instantiation from list
np_array = np.array([1, 2, 3, 4])
# Instantiation syntax for specific shapes (Crucial for ML/Analytics)
zeros_array = np.zeros(shape=(2, 3))       # 2x3 matrix of 0.0
ranged_array = np.arange(start=0, stop=10, step=2) # [0, 2, 4, 6, 8]

# Performance Difference (Vectorization)
# List requires a loop to add 1 to each element:
list_plus_one = [x + 1 for x in [1, 2, 3]]
# NumPy does it instantly in C-backend (Vectorized):
array_plus_one = np_array + 1

---

## 2. Advanced Logic: Lambda Functions, Iteration vs. Recursion

### Lambda Functions

Anonymous, single-line functions. In data roles, they are frequently used inside configuration settings, data cleaning steps (`.apply()`), or sorting logic.

In [ ]:
# Syntax: lambda arguments: expression

# Basic lambda
add_ten = lambda x: x + 10
print(add_ten(5)) # Output: 15

# Real-world Analytics Example: Sorting a list of tuples by the second element (e.g., revenue)
store_revenue = [("Store_A", 45000), ("Store_B", 62000), ("Store_C", 29000)]
store_revenue.sort(key=lambda x: x[1], reverse=True)
# Result sorted by revenue descending: [('Store_B', 62000), ('Store_A', 45000), ...]

### Iteration vs. Recursion

* **Iteration:** Uses loops (`for`, `while`) to repeat a block of code. Memory efficient ($O(1)$ space complexity) because it reuses variables.
* **Recursion:** A function that calls itself. Conceptually elegant for hierarchical data (e.g., navigating nested JSON API payloads, file directories, or corporate org charts), but hazards a `RecursionError` if it runs too deep due to call stack limits.

In [ ]:
# Scenario: Calculating a factorial (e.g., calculating permutations in data science)

# 1. Iterative Approach (Preferred for performance)
def factorial_iterative(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

# 2. Recursive Approach (Preferred for tree/hierarchical structures)
def factorial_recursive(n):
    if n == 1: # Base Case
        return 1
    else:
        return n * factorial_recursive(n - 1) # Recursive Case

---

## 3. Data Analytics Powerhouse: Pandas

Pandas is the industry standard for in-memory data manipulation, exploratory data analysis (EDA), and data cleaning on small-to-medium datasets.

In [ ]:
import pandas as pd

# --- Instantiation & Loading ---
# Load from CSV / Parquet
df = pd.read_csv('data.csv')
# df = pd.read_parquet('data.parquet')

# Instantiation from Dictionary
data = {'Employee': ['Alice', 'Bob', 'Charlie'], 'Salary': [90000, 110000, 85000], 'Dept': ['DE', 'DS', 'DE']}
df = pd.DataFrame(data)

# --- Exploratory Data Analysis (EDA) ---
df.head(5)          # View first 5 rows
df.info()            # Schema, memory usage, missing values
df.describe()        # Summary statistics (mean, max, min, etc.)

# --- Selection & Filtering ---
# Selecting columns
salaries = df['Salary']
# Filtering rows
de_team = df[df['Dept'] == 'DE']

# --- Data Cleaning ---
df['Salary'] = df['Salary'].fillna(0)            # Handle nulls
df['Employee'] = df['Employee'].apply(lambda x: x.upper()) # Apply Lambda transformation

# --- Aggregations & Group By ---
# Get average salary per department
avg_sal_by_dept = df.groupby('Dept')['Salary'].mean().reset_index()

# --- Merging & Joining ---
# df_joined = pd.merge(df1, df2, on='common_id', how='left')

---

## 4. Big Data Engineering: PySpark

When data scales past a single machine's RAM, data engineers transition from Pandas to PySpark to leverage distributed processing.

| Feature | Pandas | PySpark |
| --- | --- | --- |
| **Execution** | Eager (Computes immediately) | Lazy (Builds DAG/Plan, computes on actions) |
| **Scale** | Single Machine (Scale-up) | Distributed Cluster (Scale-out) |
| **Indices** | Has explicit row indexes | No default row indexing (conceptually a relational table) |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --- Instantiation ---
spark = SparkSession.builder \
    .appName("DataEngineeringCheatSheet") \
    .getOrCreate()

# Load Data
df = spark.read.parquet("hdfs://or/s3/path/data.parquet")
# Load CSV with schema inference
df_csv = spark.read.csv("raw_data.csv", header=True, inferSchema=True)

# --- Transformations (Lazy Evaluation) ---
# Selecting and renaming columns
df_transformed = df.select(
    F.col("Employee").alias("employee_name"),
    "Salary",
    "Dept"
)

# Filtering and creating new columns using conditions
df_filtered = df_transformed.filter(F.col("Salary") > 50000) \
                            .withColumn("is_senior", F.when(F.col("Salary") > 100000, True).otherwise(False))

# --- Aggregations & Grouping ---
# Calculate average salary by department
df_grouped = df_filtered.groupBy("Dept").agg(
    F.avg("Salary").alias("avg_salary"),
    F.count("employee_name").alias("headcount")
)

# --- Actions (Triggers actual cluster execution) ---
df_grouped.show()                 # Prints preview to console
result_list = df_grouped.collect() # Brings data back to driver node (Careful: can OOM if data is massive)

# --- Writing Data ---
# Partitioning by column is standard practice in data lakes
df_filtered.write.mode("overwrite").partitionBy("Dept").parquet("output/path/")

---

## 5. Summary Cheat Sheet: Quick Syntax Reference

In [ ]:
# List comprehension
[x for x in data if x > 0]

# Lambda for feature engineering
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x in [5,6] else 0)

# Pandas chaining syntax
df_summary = df.dropna().groupby('category').sum()

# PySpark handling nulls and defaults
df_clean = df_spark.fillna({"status": "unknown"}).filter(df_spark.date.isNotNull())